In [0]:


from pyspark.sql import functions as F
from datetime import datetime, timezone

# ─── ✏️  MUST MATCH your catalog/database ────────────────────────────────────
CATALOG        = "main"
DATABASE       = "flight_tracker"
SILVER_CLEAN   = f"{CATALOG}.{DATABASE}.flight_clean"
SILVER_CURRENT = f"{CATALOG}.{DATABASE}.flight_current"
GOLD_METRICS   = f"{CATALOG}.{DATABASE}.flight_metrics"
GOLD_ROUTES    = f"{CATALOG}.{DATABASE}.route_summary"
GOLD_HOTSPOTS  = f"{CATALOG}.{DATABASE}.airspace_hotspots"
# ─────────────────────────────────────────────────────────────────────────────

print(f"Gold tables: {GOLD_METRICS} | {GOLD_ROUTES} | {GOLD_HOTSPOTS}")

# COMMAND ----------
# MAGIC %md
# MAGIC ## Step 2: Create Gold Tables

# COMMAND ----------

spark.sql(f"""
CREATE TABLE IF NOT EXISTS {GOLD_METRICS} (
    snapshot_at     TIMESTAMP  NOT NULL,
    metric_name     STRING     NOT NULL,
    dimension       STRING,
    dimension_value STRING,
    metric_value    DOUBLE
)
USING DELTA
COMMENT 'Gold: timestamped KPI snapshots — narrow metric/dimension/value format for BI tools'
TBLPROPERTIES ('delta.autoOptimize.optimizeWrite'='true', 'delta.autoOptimize.autoCompact'='true')
""")
print(f"✅ {GOLD_METRICS}")

spark.sql(f"""
CREATE TABLE IF NOT EXISTS {GOLD_ROUTES} (
    computed_at          TIMESTAMP,
    callsign             STRING,
    airline_icao         STRING,
    aircraft_type        STRING,
    total_observations   BIGINT,
    avg_altitude         DOUBLE,
    avg_speed            DOUBLE,
    min_lat              DOUBLE,
    max_lat              DOUBLE,
    min_lon              DOUBLE,
    max_lon              DOUBLE,
    regions_seen         STRING
)
USING DELTA
COMMENT 'Gold: aggregated route profile per callsign'
TBLPROPERTIES ('delta.autoOptimize.optimizeWrite'='true')
""")
print(f"✅ {GOLD_ROUTES}")

spark.sql(f"""
CREATE TABLE IF NOT EXISTS {GOLD_HOTSPOTS} (
    computed_at      TIMESTAMP,
    grid_cell        STRING,
    lat_bucket       INT,
    lon_bucket       INT,
    flight_count     BIGINT,
    avg_altitude     DOUBLE,
    emergency_count  BIGINT,
    military_count   BIGINT
)
USING DELTA
COMMENT 'Gold: flight density heatmap by 1-degree grid cell'
TBLPROPERTIES ('delta.autoOptimize.optimizeWrite'='true')
""")
print(f"✅ {GOLD_HOTSPOTS}")

# COMMAND ----------
# MAGIC %md
# MAGIC ## Step 3: Compute KPI Snapshots → `flight_metrics`

# COMMAND ----------

def compute_gold_kpis():
    """
    Compute all KPIs from Silver active records (is_deleted=FALSE).
    Soft-deleted records are included separately for disappearance rate.
    Appends a timestamped batch to flight_metrics.
    """
    snapshot_at = datetime.now(timezone.utc).strftime("%Y-%m-%d %H:%M:%S")
    rows = []

    def scalar(name, sql):
        val = spark.sql(sql).collect()[0][0]
        rows.append((snapshot_at, name, None, None, float(val or 0)))
        print(f"   {name:35s} = {val}")

    def grouped(name, dim, sql):
        results = spark.sql(sql).collect()
        for r in results:
            rows.append((snapshot_at, name, dim, str(r[0]), float(r[1] or 0)))
        print(f"   {name:35s} = {len(results)} groups")

    print("📊 Scalar KPIs:")
    scalar("active_flights",
           f"SELECT COUNT(*) FROM {SILVER_CURRENT} WHERE is_deleted=FALSE")
    scalar("aircraft_count",
           f"SELECT COUNT(DISTINCT hex) FROM {SILVER_CURRENT} WHERE is_deleted=FALSE")
    scalar("avg_altitude",
           f"SELECT ROUND(AVG(alt_baro),0) FROM {SILVER_CURRENT} WHERE is_deleted=FALSE AND alt_baro IS NOT NULL")
    scalar("avg_speed",
           f"SELECT ROUND(AVG(gs),1) FROM {SILVER_CURRENT} WHERE is_deleted=FALSE AND gs IS NOT NULL")
    scalar("emergency_flights",
           f"SELECT COUNT(*) FROM {SILVER_CURRENT} WHERE is_deleted=FALSE AND is_emergency=TRUE")
    scalar("military_flights",
           f"SELECT COUNT(*) FROM {SILVER_CURRENT} WHERE is_deleted=FALSE AND is_military=TRUE")
    scalar("disappeared_aircraft",
           f"SELECT COUNT(*) FROM {SILVER_CURRENT} WHERE is_deleted=TRUE AND deleted_reason='DISAPPEARED'")
    scalar("disappearance_rate_pct", f"""
        SELECT ROUND(100.0 * SUM(CASE WHEN is_deleted=TRUE THEN 1 ELSE 0 END)
                           / NULLIF(COUNT(*),0), 2)
        FROM {SILVER_CURRENT}""")

    print("\n📊 Grouped KPIs:")
    grouped("flights_by_airline", "airline_icao", f"""
        SELECT COALESCE(airline_icao,'UNKNOWN'), COUNT(*)
        FROM {SILVER_CURRENT} WHERE is_deleted=FALSE
        GROUP BY 1 ORDER BY 2 DESC""")

    grouped("flights_by_type", "aircraft_type", f"""
        SELECT COALESCE(aircraft_type,'UNKNOWN'), COUNT(*)
        FROM {SILVER_CURRENT} WHERE is_deleted=FALSE
        GROUP BY 1 ORDER BY 2 DESC""")

    grouped("flights_by_region", "source_region", f"""
        SELECT source_region, COUNT(*)
        FROM {SILVER_CURRENT} WHERE is_deleted=FALSE
        GROUP BY 1 ORDER BY 2 DESC""")

    grouped("top_operators", "airline_icao", f"""
        SELECT COALESCE(airline_icao,'UNKNOWN'), COUNT(*)
        FROM {SILVER_CURRENT} WHERE is_deleted=FALSE
        GROUP BY 1 ORDER BY 2 DESC LIMIT 10""")

    grouped("route_frequency", "callsign", f"""
        SELECT callsign, COUNT(*) AS obs
        FROM {SILVER_CLEAN}
        WHERE is_deleted=FALSE AND callsign<>'UNKNOWN'
        GROUP BY 1 ORDER BY 2 DESC LIMIT 20""")

    grouped("flight_density", "grid", f"""
        SELECT CONCAT(CAST(FLOOR(lat) AS INT),'_',CAST(FLOOR(lon) AS INT)), COUNT(*)
        FROM {SILVER_CURRENT}
        WHERE is_deleted=FALSE AND lat IS NOT NULL AND lon IS NOT NULL
        GROUP BY 1 ORDER BY 2 DESC""")

    grouped("flights_by_altitude_band", "altitude_band", f"""
        SELECT CASE
            WHEN alt_baro < 0     THEN 'ground_level'
            WHEN alt_baro < 10000 THEN '0_10k_ft'
            WHEN alt_baro < 20000 THEN '10k_20k_ft'
            WHEN alt_baro < 35000 THEN '20k_35k_ft'
            ELSE '35k_plus_ft' END, COUNT(*)
        FROM {SILVER_CURRENT}
        WHERE is_deleted=FALSE AND alt_baro IS NOT NULL
        GROUP BY 1 ORDER BY 2 DESC""")

    grouped("emergency_by_squawk", "squawk", f"""
        SELECT COALESCE(squawk,'UNKNOWN'), COUNT(*)
        FROM {SILVER_CURRENT}
        WHERE is_deleted=FALSE AND is_emergency=TRUE
        GROUP BY 1 ORDER BY 2 DESC""")

    # Write to Gold
    schema = ["snapshot_at","metric_name","dimension","dimension_value","metric_value"]
    (spark.createDataFrame(rows, schema)
         .withColumn("snapshot_at", F.to_timestamp("snapshot_at"))
         .write.format("delta").mode("append").saveAsTable(GOLD_METRICS)
    )
    print(f"\n✅ flight_metrics: {len(rows):,} rows at {snapshot_at}")
    return len(rows)

compute_gold_kpis()

# COMMAND ----------
# MAGIC %md
# MAGIC ## Step 4: Route Summary → `route_summary`

# COMMAND ----------

def compute_route_summary():
    df = spark.sql(f"""
        SELECT
            callsign,
            airline_icao,
            aircraft_type,
            COUNT(*)                                    AS total_observations,
            ROUND(AVG(alt_baro), 0)                     AS avg_altitude,
            ROUND(AVG(gs), 1)                           AS avg_speed,
            MIN(lat) AS min_lat,  MAX(lat) AS max_lat,
            MIN(lon) AS min_lon,  MAX(lon) AS max_lon,
            ARRAY_JOIN(COLLECT_SET(source_region), ', ') AS regions_seen
        FROM {SILVER_CLEAN}
        WHERE is_deleted=FALSE AND callsign<>'UNKNOWN'
        GROUP BY callsign, airline_icao, aircraft_type
        ORDER BY total_observations DESC
    """).withColumn("computed_at", F.current_timestamp())

    n = df.count()
    df.write.format("delta").mode("append").saveAsTable(GOLD_ROUTES)
    print(f"✅ route_summary: {n:,} callsigns")

compute_route_summary()

# COMMAND ----------
# MAGIC %md
# MAGIC ## Step 5: Airspace Hotspots → `airspace_hotspots`

# COMMAND ----------

def compute_hotspots():
    df = spark.sql(f"""
        SELECT
            CONCAT(CAST(FLOOR(lat) AS INT),'_',CAST(FLOOR(lon) AS INT)) AS grid_cell,
            CAST(FLOOR(lat) AS INT)  AS lat_bucket,
            CAST(FLOOR(lon) AS INT)  AS lon_bucket,
            COUNT(*)                 AS flight_count,
            ROUND(AVG(alt_baro),0)   AS avg_altitude,
            SUM(CASE WHEN is_emergency=TRUE THEN 1 ELSE 0 END) AS emergency_count,
            SUM(CASE WHEN is_military=TRUE  THEN 1 ELSE 0 END) AS military_count
        FROM {SILVER_CURRENT}
        WHERE is_deleted=FALSE AND lat IS NOT NULL AND lon IS NOT NULL
        GROUP BY 1,2,3
        ORDER BY flight_count DESC
    """).withColumn("computed_at", F.current_timestamp())

    n = df.count()
    df.write.format("delta").mode("overwrite").saveAsTable(GOLD_HOTSPOTS)
    print(f"✅ airspace_hotspots: {n:,} grid cells")

compute_hotspots()

# COMMAND ----------
# MAGIC %md
# MAGIC ## Step 6: Verify Gold Tables

# COMMAND ----------

print("=" * 60)
print("GOLD LAYER VERIFICATION")
print("=" * 60)

print("\n📊 Scalar KPIs (latest snapshot)")
spark.sql(f"""
    SELECT metric_name, metric_value
    FROM {GOLD_METRICS}
    WHERE dimension IS NULL
      AND snapshot_at = (SELECT MAX(snapshot_at) FROM {GOLD_METRICS})
    ORDER BY metric_name
""").show(truncate=False)

print("\n🛫 Top 10 Busiest Routes")
spark.sql(f"""
    SELECT callsign, airline_icao, aircraft_type, total_observations,
           avg_altitude, avg_speed, regions_seen
    FROM {GOLD_ROUTES}
    ORDER BY total_observations DESC LIMIT 10
""").show(truncate=False)

print("\n🗺️  Top 10 Busiest Airspace Cells")
spark.sql(f"""
    SELECT grid_cell, lat_bucket, lon_bucket, flight_count,
           avg_altitude, emergency_count, military_count
    FROM {GOLD_HOTSPOTS}
    ORDER BY flight_count DESC LIMIT 10
""").show(truncate=False)

print("\n📅 KPI Snapshot History")
spark.sql(f"""
    SELECT DATE_TRUNC('minute', snapshot_at) AS snapshot_min,
           COUNT(DISTINCT metric_name) AS metrics
    FROM {GOLD_METRICS}
    GROUP BY 1 ORDER BY 1 DESC LIMIT 10
""").show()
